In [2]:
import os

In [3]:
%pwd

'c:\\Users\\HP\\Desktop\\swatisingh\\ml\\mlops_project\\kidney-disease-classification-project\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\HP\\Desktop\\swatisingh\\ml\\mlops_project\\kidney-disease-classification-project'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [7]:
from kidney_disease_classifier.constants import *
from kidney_disease_classifier.utils.common import read_yaml, create_directories



In [8]:
class configurationManager:
    def __init__(self, config_file_path = CONFIG_FILE_PATH, params_file_path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])




    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config

In [9]:
import os
import zipfile
import gdown
from kidney_disease_classifier import logger
from kidney_disease_classifier.utils.common import get_size

In [10]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    
    def download_file(self) -> str:
        """
        Download the data file from the source URL.
        """
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} to {zip_download_dir}")


            file_id = dataset_url.split("/")[-2]
            prefix = 'https://drive.google.com/uc?/export=download&id='
            gdown.download(prefix + file_id, zip_download_dir, quiet=False)
       
            logger.info(f"Downloaded data to {zip_download_dir}")
            
        except Exception as e:
            logger.exception(f"Error occurred while downloading data: {e}")
            raise e
        
    
    def extract_zip_file(self) -> None:
        """
        Extract the downloaded zip file to the specified directory.
        """
        unzip_Path = self.config.unzip_dir
        os.makedirs(unzip_Path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_Path)

In [11]:
try:
    config_manager = configurationManager()
    data_ingestion_config = config_manager.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    logger.exception(f"Error occurred during data ingestion: {e}")
    raise e

[2026-03-01 21:31:54,222: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-01 21:31:54,225: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-01 21:31:54,228: INFO: common: Directory created at: artifacts]
[2026-03-01 21:31:54,231: INFO: 873521965: Downloading data from https://drive.google.com/file/d/1q_Yn9A7uhPRz7ew9d51ucwgBiDIpFgR6/view?usp=drive_link to artifacts/data_ingestion/kidneyscan_data.zip]


Downloading...
From (original): https://drive.google.com/uc?/export=download&id=1q_Yn9A7uhPRz7ew9d51ucwgBiDIpFgR6
From (redirected): https://drive.google.com/uc?%2Fexport=download&id=1q_Yn9A7uhPRz7ew9d51ucwgBiDIpFgR6&confirm=t&uuid=c5a27955-021a-43b3-a749-22abaf92a116
To: c:\Users\HP\Desktop\swatisingh\ml\mlops_project\kidney-disease-classification-project\artifacts\data_ingestion\kidneyscan_data.zip
100%|██████████| 1.63G/1.63G [02:18<00:00, 11.7MB/s]

[2026-03-01 21:34:16,877: INFO: 873521965: Downloaded data to artifacts/data_ingestion/kidneyscan_data.zip]
